# Venturi Control & Disruption Experiments

The **Venturi controller** is a hierarchical hybrid stack:

- **Slow loop (~100 ms):** Bayesian rotational prior sets sweep/RMP bounds
- **Fast loop (~1 ms):** traffic-aware divertor circularization
- **Watchdog:** PID safety override on engineering limit breach

`DisruptionDetector` fuses HELIX SNR, trainable ELM predictor, and MHD stability margins into actuation recommendations.

---

## Hierarchical control as nested optimization (VISION §5)

**Slow loop** (Bayesian prior, ~100 ms): maps global equilibrium features $\mathbf{s} = (\bar{n}_e, \beta_N, f_{rot}, q_{95})$ to policy bounds $(v_{sweep}^{max}, \phi_{RMP}^{max})$.

**Fast loop** (PPO / traffic router, ~1 ms): maps divertor heat flux map $Q(\theta, \phi)$ to actuation $\mathbf{a} = (v_{sweep}, \phi_{RMP}, \dot{n}_{gas})$.

**Watchdog** (hardware): if $Q_{peak} > 0.8\, Q_{eng}$, override $\mathbf{a} \leftarrow \mathbf{a}_{safe}$.

### Circularization reward

$$\mathcal{R} = -\mathrm{Var}_t(Q_{divertor}) - 0.1\, Q_{peak} + \mathbb{1}_{\{\mathrm{Var} < 0.5\}} \cdot 5 - \mathbb{1}_{\{watchdog\}} \cdot 10$$

### Plasma traffic router (novel abstraction)

Congestion ratio $\chi = Q_{peak} / Q_{eng}$. The router treats SOL exhaust like **network load balancing** — when $\chi \to 1$, increase sweep and RMP phase to redistribute flux toroidally (analogy: TCP packets across servers).

### Disruption fusion functional

$$P_{dis} = \mathrm{clip}\big(0.45\, P_{ELM} + 0.35\, P_{MHD} + 0.2\, P_{HELIX},\, 0,\, 1\big)$$

where $P_{MHD}$ derives from $(q_{min}, \beta_N)$ stability margins — **cross-domain fusion of diagnostics, ML, and MHD** unique to fuselk.


In [ ]:
import sys
from pathlib import Path

from deepiri_fuselk.data.notebook_loaders import resolve_repo_root

repo = resolve_repo_root()

try:
    import deepiri_fuselk  # noqa: F401
except ImportError:
    sys.path.insert(0, str(repo / "src"))
    import deepiri_fuselk  # noqa: F401

import matplotlib.pyplot as plt
import numpy as np

plt.style.use("seaborn-v0_8-whitegrid")
%config InlineBackend.figure_formats = ["retina"]

from deepiri_fuselk.data.imas_loader import load_imas_hdf5
from deepiri_fuselk.data.notebook_loaders import (
    ensure_fetched_data,
    imas_to_synthetic_shot,
    list_shots,
    load_corpus_arrays,
    load_odl_meta,
    manifest_summary,
    resolve_data_root,
)

data_root = ensure_fetched_data(resolve_data_root(repo), n_shots=100, max_odl=50)
cmod_shots = list_shots(data_root, source="cmod")
syn_shots = list_shots(data_root, source="synthetic")
corpus = load_corpus_arrays(data_root)
manifest = manifest_summary(data_root)

print(f"Data lake: {data_root}")
print(f"  C-Mod (ODL) shots: {len(cmod_shots)}")
print(f"  Synthetic shots:   {len(syn_shots)}")
print(f"  ELM corpus frames: {len(corpus['labels'])} (elm_rate={corpus['elm_rate']:.2f})")
for row in manifest:
    print(f"  [{row['source']}] {row['status']} — {row['shots']} shots @ {row['fetched_at'][:19]}")


In [ ]:
from deepiri_fuselk.control.venturi_controller import VenturiController
from deepiri_fuselk.helix import HelixEngine
from deepiri_fuselk.models.disruption_detector import DisruptionDetector
from deepiri_fuselk.models.elm_predictor import ELMPredictor

venturi = VenturiController(engineering_limit=10.0)

# Slow loop priors from real C-Mod profiles
cmod = load_imas_hdf5(cmod_shots[0])
ne_ped = float(np.mean(cmod.ne_profile.values[:4]))
beta_n = float(np.mean(cmod.Te_profile.values)) / 2000.0
q95 = float(cmod.q_profile.values[-1])
prior = venturi.slow_loop(ne_pedestal=ne_ped / 1e20, beta_n=beta_n, rotation_khz=5.5, q95=q95)

# Divertor heat flux from HELIX focal map (real shot)
ece = imas_to_synthetic_shot(cmod)
helix = HelixEngine().process(ece.heat_field, ece.raw_signal, ece.angles)
heat_flux = helix.focal_map
action = venturi.fast_loop(heat_flux, prior, elm_probability=helix.elm_probability)
state = venturi.step(heat_flux, elm_probability=helix.elm_probability)

print(f"Prior max sweep: {prior.max_sweep_velocity:.2f}, max RMP: {prior.max_rmp_phase:.2f}")
print(f"Action: sweep={action.sweep_velocity:.2f}, RMP={action.rmp_phase:.2f}, gas={action.gas_puff:.1f}")
print(f"Pellet ready: {action.pellet_ready}, watchdog override: {action.overridden}")
print(f"Venturi reward: {state.reward:.3f}")


In [ ]:
# Train ELM predictor on fetched elm_corpus.npz (from `fuselk data fetch`)
elm = ELMPredictor()
acc = elm.train(corpus["features"], corpus["labels"])
detector = DisruptionDetector(elm)

# Assess every C-Mod shot with real q_min / beta_n from profiles
engine = HelixEngine()
odl_rates, dis_probs, actions = [], [], []
for path in cmod_shots:
    imas = load_imas_hdf5(path)
    ece = imas_to_synthetic_shot(imas)
    hres = engine.process(ece.heat_field, ece.raw_signal, ece.angles)
    q_min = float(np.min(imas.q_profile.values))
    beta_n = float(np.mean(imas.Te_profile.values)) / 2000.0
    assessment = detector.assess(hres, q_min=q_min, beta_n=beta_n)
    meta = load_odl_meta(path)
    odl_rates.append(meta["elm_event_rate"] if meta else 0.0)
    dis_probs.append(assessment.probability)
    actions.append(assessment.recommended_action)

print(f"ELM train accuracy (fetched corpus, n={len(corpus['labels'])}): {acc:.1%}")
print(f"Mean disruption P_dis on C-Mod corpus: {np.mean(dis_probs):.2f}")
print(f"Most common action: {max(set(actions), key=actions.count)}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(odl_rates, dis_probs, s=60, c=dis_probs, cmap="Reds", edgecolors="k", linewidths=0.3)
ax.set_xlabel("ODL density-limit phase rate")
ax.set_ylabel("Fused disruption probability P_dis")
ax.set_title("Control stack vs. experimental precursor labels")
plt.tight_layout()
plt.show()


## Experiment: heat-flux profile sweep

In [ ]:
profiles = []
rewards = []
x = np.linspace(-1, 1, 32)
X, Y = np.meshgrid(x, x)
for peak in np.linspace(2, 12, 11):
    hf = peak * np.exp(-(X**2 + Y**2) / 0.4)
    st = venturi.step(hf, elm_probability=0.3)
    profiles.append(peak)
    rewards.append(st.reward)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(profiles, rewards, "o-", color="C6")
ax.axvline(10.0, color="red", ls="--", alpha=0.6, label="engineering limit")
ax.set_xlabel("peak heat flux [MW/m² proxy]"); ax.set_ylabel("Venturi reward")
ax.set_title("Divertor circularization vs. peak load")
ax.legend()
plt.tight_layout()
plt.show()


## Optional: PPO vent training (slow — ~30 s)

Uncomment to train a strike-point sweep policy with stable-baselines3.


In [ ]:
# from deepiri_fuselk.control.rl_agent import train_vent_policy
# trained = train_vent_policy(timesteps=3000)
# print(f"Mean reward: {trained.mean_reward:.3f}, policy: {trained.policy_path}")
print("RL training cell skipped (uncomment to run)")
